# Notebook 05: Clustering & Segmentation Analysis

## Objective
Perform multi-feature clustering to segment UMKM (Micro, Small & Medium Enterprises) into
meaningful groups for two key stakeholders:

1. **Government** - Identify clusters needing priority intervention (low infrastructure,
   high risk, low survival rates) to allocate development budgets effectively.
2. **Investors** - Identify clusters with high growth potential, low competition, and good
   infrastructure for targeted investment opportunities.

**Methods Used:**
- K-Means Clustering (business feature-based segmentation)
- DBSCAN (spatial density-based clustering)
- PCA for dimensionality reduction and visualization
- Composite scoring for government priority and investment opportunity


## Section 1: Load Data & Feature Selection

We load the engineered UMKM dataset and select features relevant for clustering.
The features capture multiple dimensions of each UMKM: infrastructure access,
economic environment, competition, digital readiness, and risk exposure.

We normalize all features using StandardScaler so that no single feature dominates
the distance calculations in clustering algorithms.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.neighbors import NearestNeighbors
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Load data
df = pd.read_csv('../data/umkm_engineered.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()


In [ ]:
# Select clustering features - multi-dimensional business characteristics
clustering_features = [
    'skor_infrastruktur',
    'income_per_kapita',
    'kepadatan_penduduk',
    'jumlah_kompetitor_radius_3km',
    'omset_bulanan',
    'penetrasi_kur_pct',
    'akses_internet_pct',
    'risiko_banjir',
    'risiko_gempa',
    'has_digital_presence',
    'business_maturity'
]

X_cluster = df[clustering_features].copy()
print(f"Clustering features ({len(clustering_features)}):")
for f in clustering_features:
    print(f"  - {f}: mean={X_cluster[f].mean():.3f}, std={X_cluster[f].std():.3f}")

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)
X_scaled_df = pd.DataFrame(X_scaled, columns=clustering_features)

print(f"\nScaled data shape: {X_scaled_df.shape}")
print(f"All features now have mean~0, std~1")
print(X_scaled_df.describe().round(3))


## Section 2: Optimal K Selection

Before running K-Means, we need to determine the optimal number of clusters (k).
We use three complementary methods:

1. **Elbow Method** - Plot inertia (within-cluster sum of squares) vs k. The "elbow" point
   where adding more clusters yields diminishing returns suggests optimal k.
2. **Silhouette Score** - Measures how similar each point is to its own cluster vs other clusters.
   Higher is better (range: -1 to 1).
3. **Calinski-Harabasz Index** - Ratio of between-cluster to within-cluster dispersion.
   Higher values indicate better-defined clusters.


In [ ]:
# Evaluate different values of k
k_range = range(2, 16)
inertias = []
silhouette_scores = []
calinski_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))
    calinski_scores.append(calinski_harabasz_score(X_scaled, labels))
    print(f"k={k:2d}: Inertia={kmeans.inertia_:,.0f}, Silhouette={silhouette_scores[-1]:.4f}, Calinski-Harabasz={calinski_scores[-1]:.1f}")


In [ ]:
# Plot all three metrics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Elbow Method
axes[0].plot(list(k_range), inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0].set_ylabel('Inertia (WCSS)', fontsize=12)
axes[0].set_title('Elbow Method', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Silhouette Score
axes[1].plot(list(k_range), silhouette_scores, 'rs-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score Analysis', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Calinski-Harabasz Index
axes[2].plot(list(k_range), calinski_scores, 'g^-', linewidth=2, markersize=8)
axes[2].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[2].set_ylabel('Calinski-Harabasz Index', fontsize=12)
axes[2].set_title('Calinski-Harabasz Index', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/clustering_k_selection.png', dpi=150, bbox_inches='tight')
plt.show()

# Select optimal k based on silhouette score
optimal_k = list(k_range)[np.argmax(silhouette_scores)]
# Ensure we get at least 5 clusters for meaningful segmentation
if optimal_k < 5:
    optimal_k = 5
print(f"\nSelected optimal k = {optimal_k}")
print(f"  Silhouette Score at k={optimal_k}: {silhouette_scores[optimal_k-2]:.4f}")
print(f"  Calinski-Harabasz at k={optimal_k}: {calinski_scores[optimal_k-2]:.1f}")


## Section 3: K-Means Clustering

Apply K-Means with the selected optimal k. We visualize the clusters using PCA
(Principal Component Analysis) to project the high-dimensional feature space into 2D.

K-Means partitions data into k clusters by minimizing within-cluster variance.
Each cluster centroid represents the "average" UMKM profile for that segment.


In [ ]:
# Apply K-Means with optimal k
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10, max_iter=300)
df['cluster_kmeans'] = kmeans_final.fit_predict(X_scaled)

print(f"K-Means Clustering with k={optimal_k}")
print(f"\nCluster sizes:")
cluster_sizes = df['cluster_kmeans'].value_counts().sort_index()
for cluster_id, size in cluster_sizes.items():
    print(f"  Cluster {cluster_id}: {size:,} UMKMs ({size/len(df)*100:.1f}%)")

print(f"\nInertia: {kmeans_final.inertia_:,.0f}")
print(f"Silhouette Score: {silhouette_score(X_scaled, df['cluster_kmeans']):.4f}")


In [ ]:
# PCA 2D Visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot colored by cluster
scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=df['cluster_kmeans'],
                          cmap='tab10', alpha=0.5, s=10)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
axes[0].set_title('K-Means Clusters (PCA Projection)', fontsize=14, fontweight='bold')
plt.colorbar(scatter, ax=axes[0], label='Cluster')

# Plot centroids
centroids_pca = pca.transform(kmeans_final.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
               c='red', marker='X', s=200, edgecolors='black', linewidths=2, zorder=5)

# Cluster size bar chart
colors = plt.cm.tab10(np.linspace(0, 1, optimal_k))
axes[1].bar(range(optimal_k), cluster_sizes.values, color=colors, edgecolor='black', alpha=0.8)
axes[1].set_xlabel('Cluster', fontsize=12)
axes[1].set_ylabel('Number of UMKMs', fontsize=12)
axes[1].set_title('Cluster Sizes', fontsize=14, fontweight='bold')
axes[1].set_xticks(range(optimal_k))

plt.tight_layout()
plt.savefig('../docs/kmeans_clusters_pca.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPCA explains {pca.explained_variance_ratio_.sum()*100:.1f}% of total variance in 2D")
print(f"PC1: {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"PC2: {pca.explained_variance_ratio_[1]*100:.1f}%")


## Section 4: DBSCAN Spatial Clustering

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) identifies clusters
based on density of points. Unlike K-Means, it:
- Does not require specifying k in advance
- Can find arbitrarily shaped clusters
- Identifies noise/outlier points

We use latitude, longitude, and density-related features (kepadatan_penduduk,
jumlah_kompetitor_radius_3km) to find spatial clusters of UMKM activity.


In [ ]:
# Prepare spatial features for DBSCAN
spatial_features = ['latitude', 'longitude', 'kepadatan_penduduk', 'jumlah_kompetitor_radius_3km']
X_spatial = df[spatial_features].copy()
scaler_spatial = StandardScaler()
X_spatial_scaled = scaler_spatial.fit_transform(X_spatial)

# Use k-distance plot to determine eps
k_neighbors = 5
nn = NearestNeighbors(n_neighbors=k_neighbors)
nn.fit(X_spatial_scaled)
distances, indices = nn.kneighbors(X_spatial_scaled)

# Sort distances to k-th nearest neighbor
k_distances = np.sort(distances[:, k_neighbors-1])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(len(k_distances)), k_distances, 'b-', linewidth=1)
ax.set_xlabel('Points (sorted by distance)', fontsize=12)
ax.set_ylabel(f'{k_neighbors}-th Nearest Neighbor Distance', fontsize=12)
ax.set_title('K-Distance Plot for DBSCAN eps Selection', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Find elbow point using gradient change
gradient = np.gradient(k_distances)
elbow_idx = np.argmax(gradient > np.mean(gradient) + 2*np.std(gradient))
if elbow_idx == 0:
    elbow_idx = int(len(k_distances) * 0.9)
eps_value = k_distances[elbow_idx]
ax.axhline(y=eps_value, color='r', linestyle='--', label=f'eps = {eps_value:.3f}')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print(f"Selected eps = {eps_value:.3f} based on k-distance plot")


In [ ]:
# Apply DBSCAN
dbscan = DBSCAN(eps=eps_value, min_samples=k_neighbors)
df['cluster_dbscan'] = dbscan.fit_predict(X_spatial_scaled)

n_clusters_dbscan = len(set(df['cluster_dbscan'])) - (1 if -1 in df['cluster_dbscan'].values else 0)
n_noise = (df['cluster_dbscan'] == -1).sum()

print(f"DBSCAN Results:")
print(f"  Number of clusters: {n_clusters_dbscan}")
print(f"  Noise points: {n_noise} ({n_noise/len(df)*100:.1f}%)")
print(f"\nCluster distribution:")
dbscan_counts = df['cluster_dbscan'].value_counts().sort_index()
for cluster_id, count in dbscan_counts.head(15).items():
    label = "Noise" if cluster_id == -1 else f"Cluster {cluster_id}"
    print(f"  {label}: {count:,} ({count/len(df)*100:.1f}%)")


In [ ]:
# Visualize DBSCAN vs K-Means comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# DBSCAN on lat/lon
noise_mask = df['cluster_dbscan'] == -1
axes[0].scatter(df.loc[noise_mask, 'longitude'], df.loc[noise_mask, 'latitude'],
               c='gray', alpha=0.3, s=5, label='Noise')
non_noise = df.loc[~noise_mask]
scatter1 = axes[0].scatter(non_noise['longitude'], non_noise['latitude'],
                           c=non_noise['cluster_dbscan'], cmap='tab20', alpha=0.6, s=10)
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].set_title('DBSCAN Spatial Clusters', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper left')

# K-Means on lat/lon for comparison
scatter2 = axes[1].scatter(df['longitude'], df['latitude'],
                           c=df['cluster_kmeans'], cmap='tab10', alpha=0.6, s=10)
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].set_title('K-Means Clusters (Geographic View)', fontsize=14, fontweight='bold')
plt.colorbar(scatter2, ax=axes[1], label='Cluster')

plt.tight_layout()
plt.savefig('../docs/dbscan_vs_kmeans_spatial.png', dpi=150, bbox_inches='tight')
plt.show()

# Cross-tabulation
print("\nCross-tabulation: K-Means vs DBSCAN clusters")
ct = pd.crosstab(df['cluster_kmeans'], df['cluster_dbscan'], margins=True)
print(ct.to_string())


## Section 5: Cluster Profiling

For each K-Means cluster, we compute a comprehensive profile including:
- Mean values of all clustering features
- Dominant business type (jenis_usaha)
- Dominant location (kabupaten_kota)
- Survival rate (is_survived_3yr)
- Average potential score and revenue

We then create radar/spider charts to visually compare cluster characteristics
and assign descriptive names to each cluster based on their profiles.


In [ ]:
# Compute cluster profiles
profile_features = clustering_features + ['skor_potensi', 'is_survived_3yr', 'omset_per_karyawan',
                                           'latitude', 'longitude']

cluster_profiles = df.groupby('cluster_kmeans')[profile_features].mean()
print("Cluster Profiles (Mean Values):")
print(cluster_profiles.round(3).to_string())


In [ ]:
# Dominant jenis_usaha and kabupaten per cluster
print("\nDominant Business Type per Cluster:")
for c in range(optimal_k):
    mask = df['cluster_kmeans'] == c
    top_business = df.loc[mask, 'jenis_usaha'].mode().iloc[0]
    top_kab = df.loc[mask, 'kabupaten_kota'].mode().iloc[0]
    survival = df.loc[mask, 'is_survived_3yr'].mean()
    avg_omset = df.loc[mask, 'omset_bulanan'].mean()
    avg_potensi = df.loc[mask, 'skor_potensi'].mean()
    print(f"  Cluster {c}:")
    print(f"    Top Business: {top_business}")
    print(f"    Top Location: {top_kab}")
    print(f"    Survival Rate: {survival:.3f}")
    print(f"    Avg Omset: Rp {avg_omset:.1f} juta/bulan")
    print(f"    Avg Skor Potensi: {avg_potensi:.2f}")


In [ ]:
# Assign descriptive cluster names based on profiles
def name_clusters(profiles, df, optimal_k):
    names = {}
    for c in range(optimal_k):
        row = profiles.loc[c]
        mask = df['cluster_kmeans'] == c
        
        # Determine characteristics
        high_infra = row['skor_infrastruktur'] > profiles['skor_infrastruktur'].median()
        high_digital = row['has_digital_presence'] > profiles['has_digital_presence'].median()
        high_income = row['income_per_kapita'] > profiles['income_per_kapita'].median()
        high_risk = (row['risiko_banjir'] + row['risiko_gempa']) > (profiles['risiko_banjir'] + profiles['risiko_gempa']).median()
        high_density = row['kepadatan_penduduk'] > profiles['kepadatan_penduduk'].median()
        high_maturity = row['business_maturity'] > profiles['business_maturity'].median()
        high_competition = row['jumlah_kompetitor_radius_3km'] > profiles['jumlah_kompetitor_radius_3km'].median()
        
        if high_infra and high_digital and high_income:
            name = "Urban Digital Leaders"
        elif high_infra and high_density and high_competition:
            name = "Urban Competitive Hub"
        elif not high_infra and high_risk:
            name = "High-Risk Underserved"
        elif not high_infra and not high_income and high_maturity:
            name = "Rural Traditional Mature"
        elif high_income and not high_density:
            name = "Suburban Growth Zone"
        elif not high_infra and not high_digital:
            name = "Rural Developing"
        elif high_maturity and high_infra:
            name = "Established Urban"
        else:
            name = f"Mixed Segment {c}"
        
        names[c] = name
    
    # Ensure unique names
    seen = {}
    for k, v in names.items():
        if v in seen.values():
            names[k] = f"{v} ({k})"
        seen[k] = names[k]
    
    return names

cluster_names = name_clusters(cluster_profiles, df, optimal_k)
df['cluster_name'] = df['cluster_kmeans'].map(cluster_names)

print("Cluster Names:")
for c, name in cluster_names.items():
    size = (df['cluster_kmeans'] == c).sum()
    print(f"  Cluster {c}: '{name}' (n={size:,})")


In [ ]:
# Radar/Spider Chart for cluster comparison
# Normalize profiles to 0-1 range for radar chart
radar_features = ['skor_infrastruktur', 'income_per_kapita', 'kepadatan_penduduk',
                  'omset_bulanan', 'akses_internet_pct', 'has_digital_presence',
                  'business_maturity', 'penetrasi_kur_pct']

radar_data = cluster_profiles[radar_features].copy()
# Min-max normalize each feature
for col in radar_features:
    min_val = radar_data[col].min()
    max_val = radar_data[col].max()
    if max_val > min_val:
        radar_data[col] = (radar_data[col] - min_val) / (max_val - min_val)
    else:
        radar_data[col] = 0.5

# Create radar chart
num_vars = len(radar_features)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # Complete the circle

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
colors = plt.cm.tab10(np.linspace(0, 1, optimal_k))

for c in range(optimal_k):
    values = radar_data.loc[c].tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, color=colors[c],
            label=f"C{c}: {cluster_names[c]}")
    ax.fill(angles, values, alpha=0.1, color=colors[c])

ax.set_xticks(angles[:-1])
ax.set_xticklabels([f.replace('_', '\n') for f in radar_features], fontsize=9)
ax.set_title('Cluster Profiles - Radar Chart', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
plt.tight_layout()
plt.savefig('../docs/cluster_radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()


## Section 6: Government Priority Scoring

For government stakeholders, we compute a **priority score** for each cluster based on:
- **Low infrastructure** (higher priority for underserved areas)
- **Low income** (higher priority for economically disadvantaged areas)
- **High risk** (flood + earthquake exposure)
- **Low survival rate** (areas where businesses struggle most)

Clusters with the highest priority scores represent areas where government intervention
(KUR expansion, infrastructure development, digital training) would have the greatest impact.


In [ ]:
# Compute Government Priority Score for each cluster
govt_priority = pd.DataFrame(index=range(optimal_k))
govt_priority['cluster'] = range(optimal_k)
govt_priority['cluster_name'] = [cluster_names[c] for c in range(optimal_k)]
govt_priority['n_umkm'] = [cluster_sizes[c] for c in range(optimal_k)]

# Component scores (normalized 0-1, where 1 = highest priority)
infra_scores = cluster_profiles['skor_infrastruktur']
income_scores = cluster_profiles['income_per_kapita']
risk_scores = cluster_profiles['risiko_banjir'] + cluster_profiles['risiko_gempa']
survival_scores = cluster_profiles['is_survived_3yr']

# Invert so that LOW infra/income/survival = HIGH priority
govt_priority['low_infra_score'] = 1 - (infra_scores - infra_scores.min()) / (infra_scores.max() - infra_scores.min() + 1e-10)
govt_priority['low_income_score'] = 1 - (income_scores - income_scores.min()) / (income_scores.max() - income_scores.min() + 1e-10)
govt_priority['high_risk_score'] = (risk_scores - risk_scores.min()) / (risk_scores.max() - risk_scores.min() + 1e-10)
govt_priority['low_survival_score'] = 1 - (survival_scores - survival_scores.min()) / (survival_scores.max() - survival_scores.min() + 1e-10)

# Composite priority score (weighted average)
govt_priority['priority_score'] = (
    0.30 * govt_priority['low_infra_score'] +
    0.25 * govt_priority['low_income_score'] +
    0.25 * govt_priority['high_risk_score'] +
    0.20 * govt_priority['low_survival_score']
)

# Rank by priority
govt_priority = govt_priority.sort_values('priority_score', ascending=False).reset_index(drop=True)
govt_priority['priority_rank'] = range(1, optimal_k + 1)

print("Government Priority Ranking:")
print(govt_priority[['priority_rank', 'cluster', 'cluster_name', 'n_umkm',
                     'priority_score', 'low_infra_score', 'low_income_score',
                     'high_risk_score', 'low_survival_score']].to_string(index=False))


In [ ]:
# Recommend interventions for top priority clusters
print("\n" + "="*80)
print("GOVERNMENT INTERVENTION RECOMMENDATIONS")
print("="*80)

# Budget allocation proportional to priority score * cluster size
total_budget = 100  # Arbitrary units (e.g., billion Rupiah)
govt_priority['budget_weight'] = govt_priority['priority_score'] * govt_priority['n_umkm']
govt_priority['budget_allocation_pct'] = (govt_priority['budget_weight'] / govt_priority['budget_weight'].sum() * 100)
govt_priority['budget_allocation'] = govt_priority['budget_allocation_pct'] / 100 * total_budget

for _, row in govt_priority.iterrows():
    print(f"\n{'='*60}")
    print(f"Priority #{int(row['priority_rank'])}: {row['cluster_name']} (Cluster {int(row['cluster'])})")
    print(f"  UMKM Count: {int(row['n_umkm']):,}")
    print(f"  Priority Score: {row['priority_score']:.3f}")
    print(f"  Budget Allocation: {row['budget_allocation']:.1f}B IDR ({row['budget_allocation_pct']:.1f}%)")
    print(f"  Key Gaps:")
    
    interventions = []
    if row['low_infra_score'] > 0.5:
        interventions.append("    - Infrastructure: Road improvement, market facility development")
    if row['low_income_score'] > 0.5:
        interventions.append("    - Economic: KUR expansion, business incubation programs")
    if row['high_risk_score'] > 0.5:
        interventions.append("    - Risk Mitigation: Disaster preparedness, insurance subsidies")
    if row['low_survival_score'] > 0.5:
        interventions.append("    - Sustainability: Business mentoring, supply chain integration")
    
    if not interventions:
        interventions.append("    - Maintenance: Continue current programs, monitor progress")
    
    for i in interventions:
        print(i)

print(f"\n\nTotal Budget Allocation: {total_budget}B IDR")


## Section 7: Investment Opportunity Scoring

For investors, we compute an **investment score** for each cluster based on:
- **High growth potential** (high skor_potensi, high omset)
- **Low competition** (fewer competitors within radius)
- **Good infrastructure** (supports business operations)
- **High survival rate** (lower investment risk)

Clusters with highest investment scores represent the most attractive opportunities
for private sector investment in UMKM ecosystems.


In [ ]:
# Compute Investment Opportunity Score for each cluster
invest_opp = pd.DataFrame(index=range(optimal_k))
invest_opp['cluster'] = range(optimal_k)
invest_opp['cluster_name'] = [cluster_names[c] for c in range(optimal_k)]
invest_opp['n_umkm'] = [cluster_sizes[c] for c in range(optimal_k)]

# Component scores (normalized 0-1, where 1 = most attractive)
potensi_scores = cluster_profiles['skor_potensi']
competition_scores = cluster_profiles['jumlah_kompetitor_radius_3km']
infra_invest = cluster_profiles['skor_infrastruktur']
survival_invest = cluster_profiles['is_survived_3yr']
omset_scores = cluster_profiles['omset_bulanan']

invest_opp['growth_potential'] = (potensi_scores - potensi_scores.min()) / (potensi_scores.max() - potensi_scores.min() + 1e-10)
invest_opp['low_competition'] = 1 - (competition_scores - competition_scores.min()) / (competition_scores.max() - competition_scores.min() + 1e-10)
invest_opp['infra_quality'] = (infra_invest - infra_invest.min()) / (infra_invest.max() - infra_invest.min() + 1e-10)
invest_opp['survival_rate'] = (survival_invest - survival_invest.min()) / (survival_invest.max() - survival_invest.min() + 1e-10)
invest_opp['revenue_level'] = (omset_scores - omset_scores.min()) / (omset_scores.max() - omset_scores.min() + 1e-10)

# Composite investment score
invest_opp['investment_score'] = (
    0.25 * invest_opp['growth_potential'] +
    0.20 * invest_opp['low_competition'] +
    0.20 * invest_opp['infra_quality'] +
    0.20 * invest_opp['survival_rate'] +
    0.15 * invest_opp['revenue_level']
)

# Market size estimation (total omset in cluster)
for c in range(optimal_k):
    mask = df['cluster_kmeans'] == c
    invest_opp.loc[invest_opp['cluster'] == c, 'total_market_size_juta'] = df.loc[mask, 'omset_bulanan'].sum()
    invest_opp.loc[invest_opp['cluster'] == c, 'avg_omset_juta'] = df.loc[mask, 'omset_bulanan'].mean()

# Rank by investment attractiveness
invest_opp = invest_opp.sort_values('investment_score', ascending=False).reset_index(drop=True)
invest_opp['investment_rank'] = range(1, optimal_k + 1)

print("Investment Opportunity Ranking:")
print(invest_opp[['investment_rank', 'cluster', 'cluster_name', 'n_umkm',
                  'investment_score', 'growth_potential', 'low_competition',
                  'infra_quality', 'survival_rate']].to_string(index=False))


In [ ]:
# Generate investor pitch points
print("\n" + "="*80)
print("INVESTMENT OPPORTUNITY ANALYSIS")
print("="*80)

for _, row in invest_opp.iterrows():
    print(f"\n{'='*60}")
    print(f"Rank #{int(row['investment_rank'])}: {row['cluster_name']} (Cluster {int(row['cluster'])})")
    print(f"  Investment Score: {row['investment_score']:.3f}")
    print(f"  Market Size: Rp {row['total_market_size_juta']:,.0f} juta/month")
    print(f"  Avg Revenue per UMKM: Rp {row['avg_omset_juta']:.1f} juta/month")
    print(f"  UMKM Count: {int(row['n_umkm']):,}")
    print(f"  Survival Rate Score: {row['survival_rate']:.3f}")
    print(f"\n  Investor Pitch Points:")
    
    if row['growth_potential'] > 0.6:
        print(f"    + High growth potential area with strong skor_potensi")
    if row['low_competition'] > 0.6:
        print(f"    + Low competition density - first-mover advantage")
    if row['infra_quality'] > 0.6:
        print(f"    + Good infrastructure supporting business operations")
    if row['survival_rate'] > 0.6:
        print(f"    + High business survival rate - lower investment risk")
    if row['revenue_level'] > 0.6:
        print(f"    + Strong existing revenue base")
    
    # Risk factors
    risks = []
    if row['low_competition'] < 0.3:
        risks.append("High competition")
    if row['infra_quality'] < 0.3:
        risks.append("Poor infrastructure")
    if row['survival_rate'] < 0.3:
        risks.append("Low survival rate")
    if risks:
        print(f"  Risk Factors: {', '.join(risks)}")


## Section 8: Geographic Visualization

Visualize the K-Means clusters on a geographic scatter plot using latitude and longitude.
This helps identify spatial patterns in cluster distribution and potential geographic
concentration of specific cluster types.


In [ ]:
# Geographic scatter plot colored by K-Means cluster
fig, ax = plt.subplots(figsize=(14, 10))

colors_map = plt.cm.tab10(np.linspace(0, 1, optimal_k))
for c in range(optimal_k):
    mask = df['cluster_kmeans'] == c
    ax.scatter(df.loc[mask, 'longitude'], df.loc[mask, 'latitude'],
              c=[colors_map[c]], alpha=0.5, s=15,
              label=f"C{c}: {cluster_names[c]} (n={mask.sum():,})")

ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('UMKM Clusters - Geographic Distribution', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=9, markerscale=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/clusters_geographic.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary statistics by geography
print("\nGeographic spread per cluster:")
for c in range(optimal_k):
    mask = df['cluster_kmeans'] == c
    lat_range = df.loc[mask, 'latitude'].max() - df.loc[mask, 'latitude'].min()
    lon_range = df.loc[mask, 'longitude'].max() - df.loc[mask, 'longitude'].min()
    n_kab = df.loc[mask, 'kabupaten_kota'].nunique()
    print(f"  Cluster {c} ({cluster_names[c]}): {n_kab} kabupaten, lat range={lat_range:.3f}, lon range={lon_range:.3f}")


## Section 9: Save Results

Save all outputs for downstream use:
1. **umkm_clustered.csv** - Full dataset with cluster labels
2. **cluster_profiles.csv** - Summary statistics per cluster
3. **government_priority_clusters.csv** - Government priority ranking and recommendations
4. **investment_opportunity_clusters.csv** - Investment opportunity ranking and metrics


In [ ]:
# Save 1: Full dataset with cluster labels
output_cols = list(df.columns)
df.to_csv('../data/umkm_clustered.csv', index=False)
print(f"Saved: ../data/umkm_clustered.csv ({df.shape[0]} rows, {df.shape[1]} cols)")
print(f"  New columns: cluster_kmeans, cluster_dbscan, cluster_name")

# Save 2: Cluster profiles
profiles_save = cluster_profiles.copy()
profiles_save['cluster_name'] = [cluster_names[c] for c in profiles_save.index]
profiles_save['n_umkm'] = [cluster_sizes[c] for c in profiles_save.index]
profiles_save.index.name = 'cluster_id'
profiles_save.to_csv('../data/cluster_profiles.csv')
print(f"\nSaved: ../data/cluster_profiles.csv ({len(profiles_save)} clusters)")

# Save 3: Government priority clusters
govt_save = govt_priority[['priority_rank', 'cluster', 'cluster_name', 'n_umkm',
                           'priority_score', 'low_infra_score', 'low_income_score',
                           'high_risk_score', 'low_survival_score',
                           'budget_allocation_pct', 'budget_allocation']].copy()
govt_save.to_csv('../data/government_priority_clusters.csv', index=False)
print(f"\nSaved: ../data/government_priority_clusters.csv ({len(govt_save)} clusters)")

# Save 4: Investment opportunity clusters
invest_save = invest_opp[['investment_rank', 'cluster', 'cluster_name', 'n_umkm',
                          'investment_score', 'growth_potential', 'low_competition',
                          'infra_quality', 'survival_rate', 'revenue_level',
                          'total_market_size_juta', 'avg_omset_juta']].copy()
invest_save.to_csv('../data/investment_opportunity_clusters.csv', index=False)
print(f"\nSaved: ../data/investment_opportunity_clusters.csv ({len(invest_save)} clusters)")

print("\n" + "="*60)
print("ALL OUTPUTS SAVED SUCCESSFULLY")
print("="*60)


## Summary

This notebook performed comprehensive clustering and segmentation analysis:

1. **K-Means Clustering** identified distinct UMKM segments based on business characteristics
2. **DBSCAN** revealed spatial density patterns and geographic clusters
3. **Cluster Profiling** provided descriptive names and characteristics for each segment
4. **Government Priority Scoring** ranked clusters for intervention priority
5. **Investment Opportunity Scoring** identified the most attractive segments for investors

### Key Outputs:
- `data/umkm_clustered.csv` - Dataset with cluster assignments
- `data/cluster_profiles.csv` - Detailed cluster statistics
- `data/government_priority_clusters.csv` - Priority ranking for government programs
- `data/investment_opportunity_clusters.csv` - Investment attractiveness ranking

These outputs enable targeted decision-making by both government agencies and private investors.
